Sentinel tif files from S3 bucket cover the entirety of the US. This code was used to trim sentinel tif images to just the Huron County, Michigan area. This trimming was performed for tif files 2019 - 2023.

In [1]:
pip install --user numpy==1.23.5 opencv-python==4.8.0.76 matplotlib==3.7.1 rasterio pillow

In [2]:
import boto3
import botocore
from botocore import UNSIGNED
from botocore.config import Config
from itertools import product
from botocore.config import Config

import rasterio
import numpy as np
import cv2
import matplotlib.pyplot as plt


from rasterio.plot import reshape_as_image
from PIL import Image
import os
import json
import math

In [9]:
# Set up parameters
bucket_name = "sentinel-cogs"
base_prefix = "sentinel-s2-l2a-cogs"
# Initialize S3 client
# Initialize S3 client with unsigned config
config = Config(
    signature_version=botocore.UNSIGNED,
    retries = dict(
        max_attempts = 3
    )
)
s3 = boto3.client('s3', config=config)

# Generate all possible tiles for Huron County, MI
# We will only consider UTM zones 
# and latitude bands that cover Huron County, MI
# UTM zones for Huron County, MI: 17
# Latitude bands: T
# Reference: 
# https://www.usgs.gov/media/images/mapping-utm-grid-conterminous-48-united-states#:~:text=The%20Universal%20Transverse%20Mercator%20grid,Zone%2019%20in%20New%20England.
# https://earth-info.nga.mil/index.php?dir=coordsys&action=coordsys

utm = 17
latitude_band = 'T'  # Sentinel-2 latitude bands

# Discovery the grid squares
grid_squares_found = set()

# s3://sentinel-cogs/sentinel-s2-l2a-cogs/17/T/
# 17 is UTM, T is latitude band
# List initial contents
prefix = f"{base_prefix}/{utm}/{latitude_band}/"
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=prefix,
    Delimiter='/'
)

# # Process each grid square prefix
# for prefix in response.get('CommonPrefixes', []):
#     grid_square = prefix.get('Prefix').split('/')[3]
#     grid_squares_found.add(grid_square)
                
# print("grid squares: ", ', '.join(sorted(grid_squares_found)))

In [ ]:
# s3://sentinel-cogs/sentinel-s2-l2a-cogs/17/T/grid_square/year/month/

# we need 2019 to 2023, we need june to july
# grid square: LJ

huron_county_prefix = 'sentinel-s2-l2a-cogs/17/T/LJ/'
years = [2019, 2020, 2021, 2022, 2023]
months = [6, 7]  # June and July


response = s3.list_objects_v2(Bucket=bucket_name, Prefix=huron_county_prefix, Delimiter='/')
common_prefix = response.get('CommonPrefixes', [])

# construct the paths
year_paths = []
for prefix in common_prefix:
    prefix_str = prefix.get('Prefix')
    prefix_parts = prefix_str.strip('/').split('/') 
    year = int(prefix_parts[-1])
    if year in years:
        for month in months:
            prefix = f"{prefix_str}{month}/"
            year_paths.append(prefix)

# print(year_paths)

# get all subdirectories
subdirectories = []

for path in year_paths:
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=path, Delimiter='/')
    common_prefixes = response.get('CommonPrefixes', [])

    for prefix in common_prefixes:
        sub_prefix = prefix.get('Prefix')
        # print(sub_prefix)
        subdirectories.append(sub_prefix)

# print(subdirectories)


saved_tif_files = []
for subdirectory in subdirectories:
    json_filename = f"{subdirectory.rstrip('/')}/{subdirectory.rstrip('/').split('/')[-1]}.json"
    print(json_filename)
    response = s3.get_object(Bucket=bucket_name, Key=json_filename)
    json_content = response['Body'].read().decode('utf-8')
    metadata = json.loads(json_content)

    # Check properties
    cloud_cover = metadata.get('properties', {}).get('eo:cloud_cover')
    sun_elevation = metadata.get('properties', {}).get('view:sun_elevation')
    cloud_shadow = metadata.get('properties', {}).get('s2:cloud_shadow_percentage')
    no_data_pixel_percentage = metadata.get('properties', {}).get('s2:nodata_pixel_percentage')
    defective_pixel_percentage = metadata.get('properties', {}).get('s2:saturated_defective_pixel_percentage')

    if (cloud_cover is not None and cloud_cover < 10 and
    sun_elevation is not None and sun_elevation > 60 and
    cloud_shadow is not None and math.floor(cloud_shadow) == 0 and
    no_data_pixel_percentage is not None and math.floor(no_data_pixel_percentage) == 0 and
    defective_pixel_percentage is not None and math.floor(defective_pixel_percentage) == 0):
        tif_key = f"{subdirectory.rstrip('/')}/TCI.tif"
        local_filename = tif_key.split('/')[-2] + "_TCI.tif"
        s3.download_file(bucket_name, tif_key, local_filename)

# to find the best tif file from each year we will adjust the cloud cover and sun elevation number
# we will start with best cloud cover and sun elevation number
# we will slowly increase the cloud cover and decrease the sun elevation number if certain year does not have the tif
# file that fits the criteria

# cloud_cover < 1, sun_elevation > 80, math.floor(cloud_shadow) == 0, 
# math.floor(no_data_pixel_percentage) == 0, math.floor(defective_pixel_percentage) == 0 
# S2A_17TLJ_20190714_0_L2A_TCI.tif 
# S2A_17TLJ_20200728_0_L2A_TCI.tif 
# S2A_17TLJ_20220618_0_L2A_TCI.tif 
# S2A_17TLJ_20220628_0_L2A_TCI.tif
# S2B_17TLJ_20220623_0_L2A_TCI.tif

In [5]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask

# File paths
# 2023 sentinel and shape file
sentinel_image_path = "../data/Sentinel-2/cloud-cover-5-sun-60/S2B_17TLJ_20230618_0_L2A_TCI.tif"
shapefile_path = "../data/tl_2023_26063_edges/tl_2023_26063_edges.shp"

# Load the shapefile
gdf = gpd.read_file(shapefile_path)

# Filter for Huron County
huron_county = gdf[gdf['COUNTYFP'] == '063']

# Open Sentinel-2 image
with rasterio.open(sentinel_image_path) as src:
    # Check if CRS matches and reproject if needed
    if huron_county.crs != src.crs:
        huron_county = huron_county.to_crs(src.crs)
    
    # Mask the raster with the shapefile
    out_image, out_transform = mask(src, huron_county.geometry, crop=True)
    
    # Create a metadata dictionary for the output
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })
    
    # Save the masked/trimmed image
    output_path = "../src/trimmed-sentinel-2-data/2023.tif"
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(out_image)